In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import Module, Linear, ReLU, Sequential, ModuleList
from torch_geometric.nn import TransformerConv, GatedGraphConv, BatchNorm, LayerNorm
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import AddRandomWalkPE
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import pickle
import os
import zipfile
import torch_geometric
import math

# Initialize CUDA context explicitly
if torch.cuda.is_available():
    torch.cuda.init()
    print(f"CUDA initialized. Available GPUs: {torch.cuda.device_count()}")

# Print torch_geometric version
print(f"torch_geometric version: {torch_geometric.__version__}")

# Fix random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device for multi-GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
else:
    print("Using 1 GPU or CPU")

CUDA initialized. Available GPUs: 2
torch_geometric version: 2.6.1
Using 2 GPUs!


In [2]:
# Extract dataset
zip_path = 'METR-LA.zip'
data_dir = 'METR-LA'
if not os.path.exists(data_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
print(f"Data directory {data_dir} ready")

Data directory METR-LA ready


In [3]:
def load_adjacency_matrix(adj_matrix_path, data_dir='METR-LA'):
    try:
        with open(adj_matrix_path, 'rb') as f:
            adj_matrix = pickle.load(f)
        
        if isinstance(adj_matrix, tuple):
            for i in range(len(adj_matrix)):
                temp_matrix = np.array(adj_matrix[i])
                if temp_matrix.ndim == 2 and np.issubdtype(temp_matrix.dtype, np.number):
                    adj_matrix = temp_matrix
                    break
            else:
                raise ValueError(f"No 2D numeric array found in tuple: {adj_matrix}")
        elif isinstance(adj_matrix, dict):
            possible_keys = ['adj', 'adj_mx', 'matrix', 'adj_matrix', 'connectivity']
            for key in possible_keys:
                if key in adj_matrix:
                    adj_matrix = adj_matrix[key]
                    break
            else:
                raise ValueError(f"Adjacency matrix not found in dict. Keys found: {list(adj_matrix.keys())}")
        elif isinstance(adj_matrix, list):
            adj_matrix = np.array(adj_matrix)
        elif not isinstance(adj_matrix, np.ndarray):
            raise ValueError(f"Unexpected type for adjacency matrix: {type(adj_matrix)}")
        
        adj_matrix = np.array(adj_matrix, dtype=np.float32)
        adj_matrix = (adj_matrix + adj_matrix.T) / 2  # Symmetrize
        
        if adj_matrix.ndim != 2:
            raise ValueError(f"Expected 2D adjacency matrix, got shape {adj_matrix.shape}")
        if adj_matrix.shape[0] != adj_matrix.shape[1]:
            raise ValueError(f"Expected square adjacency matrix, got shape {adj_matrix.shape}")
        if adj_matrix.shape[0] != 207:
            raise ValueError(f"Expected 207x207 adjacency matrix for METR-LA, got shape {adj_matrix.shape}")
        
        if not np.issubdtype(adj_matrix.dtype, np.number):
            raise ValueError(f"Adjacency matrix contains non-numeric values, dtype: {adj_matrix.dtype}")
        
        non_zero_count = np.count_nonzero(adj_matrix)
        print(f"Number of non-zero entries in adjacency matrix: {non_zero_count}")
        if non_zero_count == 0:
            raise ValueError("Adjacency matrix has no non-zero entries, cannot create edge_index")
        
        print(f"Loaded adjacency matrix from {adj_matrix_path}, shape: {adj_matrix.shape}, dtype: {adj_matrix.dtype}")
        return adj_matrix
    
    except Exception as e:
        print(f"Warning: Failed to load adjacency matrix from {adj_matrix_path}: {str(e)}")
        print(f"Falling back to default adjacency matrix: {data_dir}/adj_mat.npy")
        adj_matrix = np.load(os.path.join(data_dir, 'adj_mat.npy'))
        non_zero_count = np.count_nonzero(adj_matrix)
        print(f"Loaded default adjacency matrix from {data_dir}/adj_mat.npy, shape: {adj_matrix.shape}, dtype: {adj_matrix.dtype}, non-zero entries: {non_zero_count}")
        return adj_matrix

In [4]:
def load_metr_la_data(data_dir='METR-LA', num_days=99, custom_adj_path=None):
    if custom_adj_path is not None:
        print(f"Loading custom adjacency matrix from: {custom_adj_path}")
        adj_mat = load_adjacency_matrix(custom_adj_path, data_dir)
    else:
        adj_mat = np.load(os.path.join(data_dir, 'adj_mat.npy'))
        non_zero_count = np.count_nonzero(adj_mat)
        print(f"Loaded default adjacency matrix from {data_dir}/adj_mat.npy, non-zero entries: {non_zero_count}")
    
    print(f"Adjacency matrix shape: {adj_mat.shape}, dtype: {adj_mat.dtype}")

    node_values = np.load(os.path.join(data_dir, 'node_values.npy'))
    print(f"Original node_values shape: {node_values.shape}")

    time_steps = num_days * 288
    node_values = node_values[:time_steps]
    if node_values.ndim == 3:
        node_values = node_values[:, :, 0]
        node_values = node_values.T
    elif node_values.ndim != 2:
        raise ValueError(f"Expected 2D or 3D node_values, got shape {node_values.shape}")

    print(f"Processed node_values shape: {node_values.shape}")

    # Store original shape for time feature generation
    original_node_values_shape = node_values.shape 

    scaler = MinMaxScaler()
    node_values = scaler.fit_transform(node_values.T).T

    edge_index = np.where(adj_mat > 0)
    print(f"Raw edge_index from np.where: {len(edge_index)} arrays, shapes: {[arr.shape for arr in edge_index]}")
    if len(edge_index) != 2:
        raise ValueError(f"Expected edge_index to have 2 arrays (source, target), got {len(edge_index)}")
    edge_index = np.stack(edge_index, axis=0)
    print(f"Stacked edge_index shape: {edge_index.shape}")
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    if edge_index.shape[0] != 2:
        raise ValueError(f"Expected edge_index to have shape [2, num_edges], got shape {edge_index.shape}")
    if edge_index.shape[1] == 0:
        raise ValueError("edge_index is empty; adjacency matrix may have no non-zero entries")
    edge_weight = torch.tensor(adj_mat[edge_index[0], edge_index[1]], dtype=torch.float)

    edge_index = edge_index.to(device)
    edge_weight = edge_weight.to(device)

    print(f"edge_index device: {edge_index.device}, shape: {edge_index.shape}")
    print(f"edge_weight device: {edge_weight.device}, shape: {edge_weight.shape}")

    return node_values, edge_index, edge_weight, scaler, original_node_values_shape

In [5]:
def add_time_features(features, current_time_step_index, num_intervals_per_day=288, in_timestep=12):
    """
    Adds cyclical time-of-day features to the input features.
    features: (batch_size, in_timestep, num_nodes)
    current_time_step_index: The starting global time step index for this batch.
    """
    batch_size, current_in_timestep, num_nodes = features.shape

    # Generate time-of-day features for each time step within the input window
    time_features_list = []
    for i in range(current_in_timestep):
        # Calculate the global time step for the current point in the input window
        global_time_step = (current_time_step_index + i) % num_intervals_per_day
        
        # Calculate sine and cosine components for the time of day
        sin_time = math.sin(2 * math.pi * global_time_step / num_intervals_per_day)
        cos_time = math.cos(2 * math.pi * global_time_step / num_intervals_per_day)
        
        # These features are constant across all nodes at a given time step
        # Expand them to match the number of nodes
        time_features = torch.tensor([[sin_time, cos_time]] * num_nodes, dtype=torch.float32)
        time_features_list.append(time_features)

    # Stack them to get shape (in_timestep, num_nodes, 2)
    time_features_tensor = torch.stack(time_features_list, dim=0)

    # Expand features to (batch_size, in_timestep, num_nodes, features_per_node)
    # The original features are (batch_size, in_timestep, num_nodes) representing traffic speed.
    # We need to unsqueeze the last dimension to make it (batch_size, in_timestep, num_nodes, 1)
    # so we can concatenate with time_features_tensor which is (in_timestep, num_nodes, 2).
    # Then we expand time_features_tensor to (batch_size, in_timestep, num_nodes, 2).
    
    # Repeat time_features_tensor for each sample in the batch
    time_features_batch = time_features_tensor.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    
    # Reshape original features to (batch_size, in_timestep, num_nodes, 1)
    features_reshaped = features.unsqueeze(-1)

    # Concatenate along the last dimension
    # Resulting shape: (batch_size, in_timestep, num_nodes, 1 + 2) = (batch_size, in_timestep, num_nodes, 3)
    features_with_time = torch.cat((features_reshaped, time_features_batch), dim=-1)

    return features_with_time


def create_data_samples(node_values, in_timestep=12, out_timestep=12, total_time_steps_for_time_features=None):
    features_list, labels_list = [], []
    num_nodes, total_time_steps = node_values.shape
    
    # If not provided, assume total_time_steps for time feature calculation
    if total_time_steps_for_time_features is None:
        total_time_steps_for_time_features = total_time_steps

    for i in range(total_time_steps - in_timestep - out_timestep + 1):
        # Original traffic speed features (num_nodes, in_timestep)
        speed_features = node_values[:, i:i + in_timestep]
        
        # Labels are just traffic speed (num_nodes, out_timestep)
        labels = node_values[:, i + in_timestep:i + in_timestep + out_timestep]
        
        features_list.append(speed_features)
        labels_list.append(labels)
        
    features = np.array(features_list, dtype=np.float32) # (num_samples, num_nodes, in_timestep)
    labels = np.array(labels_list, dtype=np.float32)     # (num_samples, num_nodes, out_timestep)

    # Permute to (num_samples, in_timestep, num_nodes) for features
    # and (num_samples, out_timestep, num_nodes) for labels
    features = torch.tensor(features).permute(0, 2, 1) 
    labels = torch.tensor(labels).permute(0, 2, 1)

    # Now add time features to `features`. This needs to be done sample by sample
    # as time features depend on the global index `i`.
    
    features_with_time_list = []
    for i in range(features.shape[0]):
        # `features[i]` has shape (in_timestep, num_nodes)
        # `i` here is the *sample index*, which corresponds to the global time step `i`
        # in the original `node_values` array, plus the in_timestep offset.
        
        # Correct global time step calculation for the starting point of the input window
        global_start_time_step_for_sample = i # Since `i` in the loop above is the start of the window
        
        # We need to add time features for each node at each of the `in_timestep` time steps
        # The `add_time_features` expects (batch_size, in_timestep, num_nodes)
        # So we pass features[i].unsqueeze(0) to simulate a batch of 1
        feature_sample_with_time = add_time_features(
            features[i].unsqueeze(0), 
            global_start_time_step_for_sample, 
            num_intervals_per_day=288,
            in_timestep=in_timestep
        ).squeeze(0) # Remove the batch dimension of 1
        
        features_with_time_list.append(feature_sample_with_time)

    # Stack the list to get (num_samples, in_timestep, num_nodes, in_channels)
    features_with_time = torch.stack(features_with_time_list, dim=0)

    # For `Data` object, `x` expects shape (num_nodes, in_channels) for each sample.
    # So the overall features will be (num_samples, num_nodes, in_timestep, in_channels_after_time_features)
    # We will need to reshape this correctly when creating the `Data` objects in the main loop.
    # The `x` passed to the model will be (num_nodes * batch_size, in_timestep * in_channels_after_time_features)
    # or (num_nodes, in_timestep * in_channels_after_time_features) for each data object within the batch.
    
    # Let's keep features_with_time as (num_samples, in_timestep, num_nodes, new_in_channels)
    # and then permute to (num_samples, num_nodes, in_timestep, new_in_channels)
    # So when we create Data(x=train_features[i].T), it will be (num_nodes, in_timestep * new_in_channels)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)
    
    return features_final, labels

def create_100th_day_data(data_dir='METR-LA', in_timestep=12, out_timestep=12):
    node_values_full = np.load(os.path.join(data_dir, 'node_values.npy'))
    
    # The 100th day starts at index 288 * 99 = 28512
    # So time_steps for 100th day are from 28512 to 28512 + 288 - 1
    start_global_time_index = 288 * 99 
    node_values = node_values_full[start_global_time_index : start_global_time_index + 288]
    node_values = node_values[:, :, 0].T # Shape (num_nodes, 288)

    scaler_100 = MinMaxScaler()
    node_values_scaled = scaler_100.fit_transform(node_values.T).T # Shape (num_nodes, 288)

    features_list, labels_list = [], []
    num_nodes, total_time_steps_100 = node_values_scaled.shape

    for i in range(total_time_steps_100 - in_timestep - out_timestep + 1):
        speed_features = node_values_scaled[:, i:i + in_timestep]
        labels = node_values_scaled[:, i + in_timestep:i + in_timestep + out_timestep]
        
        features_list.append(speed_features)
        labels_list.append(labels)
        
    features = np.array(features_list, dtype=np.float32)
    labels = np.array(labels_list, dtype=np.float32)

    # Permute to (num_samples, in_timestep, num_nodes)
    features = torch.tensor(features).permute(0, 2, 1)
    labels = torch.tensor(labels).permute(0, 2, 1)

    features_with_time_list = []
    for i in range(features.shape[0]):
        # Global time step for 100th day data needs to account for previous 99 days
        global_start_time_step_for_sample = start_global_time_index + i
        
        feature_sample_with_time = add_time_features(
            features[i].unsqueeze(0), 
            global_start_time_step_for_sample, 
            num_intervals_per_day=288,
            in_timestep=in_timestep
        ).squeeze(0)
        
        features_with_time_list.append(feature_sample_with_time)

    features_with_time = torch.stack(features_with_time_list, dim=0)
    features_final = features_with_time.permute(0, 2, 1, 3).reshape(features_with_time.shape[0], num_nodes, -1)

    return features_final, labels, scaler_100

In [6]:
def compute_metrics(pred, target, scaler):
    batch_size, num_nodes, out_timestep = pred.shape
    pred_np = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_np = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    pred_unscaled = scaler.inverse_transform(pred_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_np).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    pred_unscaled = torch.tensor(pred_unscaled, device=pred.device)
    target_unscaled = torch.tensor(target_unscaled, device=target.device)
    mae = torch.mean(torch.abs(pred_unscaled - target_unscaled)).item()
    rmse = torch.sqrt(torch.mean((pred_unscaled - target_unscaled) ** 2)).item()
    mask = target_unscaled > 1.0
    mape = torch.where(mask, torch.abs((pred_unscaled - target_unscaled) / (target_unscaled + 1e-2)), 0.0)
    mape = torch.mean(mape[mask]) * 100 if mask.sum() > 0 else 0.0
    return mae, rmse, mape

def visualize_predictions(pred, target, scaler, output_dir='plots', num_sensors=5, filename='prediction_100th_day.png'):
    os.makedirs(output_dir, exist_ok=True)
    
    # Ensure pred and target have the correct shape before transposing for inverse_transform
    # Expected shape for inverse_transform: (n_samples, n_features) where n_features is num_nodes
    # Current pred/target shape: (batch_size, num_nodes, out_timestep)
    # We need to reshape to (batch_size * out_timestep, num_nodes) for scaler
    
    batch_size, num_nodes, out_timestep = pred.shape
    
    pred_reshaped = pred.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    target_reshaped = target.cpu().numpy().transpose(0, 2, 1).reshape(-1, num_nodes)
    
    pred_unscaled = scaler.inverse_transform(pred_reshaped).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    target_unscaled = scaler.inverse_transform(target_reshaped).reshape(batch_size, out_timestep, num_nodes).transpose(0, 2, 1)
    
    # Now we want to plot for a single sample (the first one in the batch)
    pred_to_plot = pred_unscaled[0].T # Shape (out_timestep, num_nodes) -> (num_nodes, out_timestep)
    target_to_plot = target_unscaled[0].T # Shape (out_timestep, num_nodes) -> (num_nodes, out_timestep)

    plt.figure(figsize=(12, 10))
    for i in range(min(num_sensors, pred_to_plot.shape[0])):# Iterate over nodes (sensors)
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(pred_to_plot[i], label='Predicted', marker='o', markersize=3)
        plt.plot(target_to_plot[i], label='Actual', marker='x', markersize=3)
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (Sample)')
        plt.xlabel('Time Step (5-min intervals)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

def visualize_full_100th_day(day_100_loader, model, scaler, output_dir='plots', num_sensors=5, filename='full_100th_day_prediction.png'):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            all_preds.append(out.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    num_nodes = all_preds.shape[1]
    
    # 288 is the number of 5-minute intervals in a day
    full_preds_accum = np.zeros((num_nodes, 288)) 
    full_targets_accum = np.zeros((num_nodes, 288))
    counts = np.zeros((num_nodes, 288))

    # `all_preds` and `all_targets` have shape (num_samples, num_nodes, out_timestep)
    # Each sample corresponds to a prediction starting from a certain 5-minute interval.
    # The output `out_timestep` is 12.
    
    # We need to correctly place these predictions into the full 288-interval day.
    # The `create_100th_day_data` generates samples starting from index `i` of the day.
    # The labels are `i + in_timestep` to `i + in_timestep + out_timestep`.
    # So predictions `out` correspond to labels `target` at these indices.
    
    # `in_timestep` is 12.
    # `out_timestep` is 12.
    
    for i in range(all_preds.shape[0]):
        # The prediction for sample `i` covers the time steps from `i + in_timestep` to `i + in_timestep + out_timestep - 1`
        start_idx_in_day = i + 12 
        end_idx_in_day = start_idx_in_day + 12 
        
        # Add predictions/targets to the accumulated arrays
        full_preds_accum[:, start_idx_in_day:end_idx_in_day] += all_preds[i]
        full_targets_accum[:, start_idx_in_day:end_idx_in_day] += all_targets[i]
        counts[:, start_idx_in_day:end_idx_in_day] += 1

    # Avoid division by zero for unpredicted parts (usually edges)
    mask = counts > 0
    full_preds_mean = np.zeros_like(full_preds_accum)
    full_targets_mean = np.zeros_like(full_targets_accum)
    
    full_preds_mean[mask] = full_preds_accum[mask] / counts[mask]
    full_targets_mean[mask] = full_targets_accum[mask] / counts[mask]

    # Inverse transform. Scaler expects (num_samples, num_features)
    # Here, num_samples is 288 time steps, num_features is num_nodes.
    full_preds_unscaled = scaler.inverse_transform(full_preds_mean.T).T
    full_targets_unscaled = scaler.inverse_transform(full_targets_mean.T).T

    # Only plot the predicted range, which is roughly from 12 to 276 (288 - 12)
    # The actual covered time steps are from 12 to (288 - out_timestep) = 288 - 12 = 276
    covered_time_steps = slice(12, 288 - 12) 
    time_steps = np.arange(12, 288 - 12) * 5 / 60 # Convert 5-min intervals to hours

    os.makedirs(output_dir, exist_ok=True)
    plt.figure(figsize=(15, 10))
    for i in range(min(num_sensors, num_nodes)):
        plt.subplot(num_sensors, 1, i + 1)
        plt.plot(time_steps, full_preds_unscaled[i, covered_time_steps], label='Predicted', marker='o', markersize=3)
        plt.plot(time_steps, full_targets_unscaled[i, covered_time_steps], label='Actual', marker='x', markersize=3)
        plt.title(f'Sensor {i + 1} Traffic Speed Prediction (100th Day, Hours {time_steps[0]:.0f}-{time_steps[-1]:.0f})')
        plt.xlabel('Time (hours)')
        plt.ylabel('Speed (mph)')
        plt.legend()
        plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

In [7]:
class GPSLayer(nn.Module):
    def __init__(self, hidden_channels, heads=8, dropout=0.1):
        super(GPSLayer, self).__init__()
        self.gated_gcn = GatedGraphConv(hidden_channels, num_layers=1)
        self.transformer = TransformerConv(
            hidden_channels, hidden_channels, heads=heads, concat=False, dropout=dropout
        )
        self.norm1 = LayerNorm(hidden_channels)
        self.norm2 = LayerNorm(hidden_channels)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index, edge_weight, batch):
        # x shape: (num_nodes * batch_size, hidden_channels)
        # Gated GCN path
        x_gcn = self.gated_gcn(x, edge_index, edge_weight=edge_weight)
        x_gcn = self.norm1(x_gcn)
        
        # Transformer path
        x_trans = self.transformer(x_gcn, edge_index)
        
        # Residual connection
        x = x_gcn + x_trans + x
        x = self.norm2(x)
        x = torch.relu(x)
        x = self.dropout(x)
        
        # Feed-forward network
        x_ffn = self.ffn(x)
        x = x + x_ffn
        return x

class GPSGraphTransformer(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=12, num_layers=3, pe_dim=20, heads=4, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        
        # The input `x` to the model is now (num_nodes, in_timestep * original_in_channels)
        # where original_in_channels is (1 for speed + 2 for time_of_day) = 3
        # So effective in_channels for Linear layer needs to be in_timestep * (1 + 2)
        
        self.node_emb = nn.Linear(in_channels, hidden_channels - pe_dim if use_pe else hidden_channels)
        
        if use_pe:
            self.pe_lin = nn.Linear(pe_dim, pe_dim)
            
        self.layers = nn.ModuleList([GPSLayer(hidden_channels, heads=heads, dropout=0.1) for _ in range(num_layers)])
        self.norm = LayerNorm(hidden_channels)
        
        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_channels // 2, out_channels)
        )
    
    def forward(self, x, edge_index, edge_weight, pe=None, batch=None):
        # x shape for Data object is (num_nodes, in_timestep * in_features_per_timestep)
        # We need to reshape x to (num_nodes, features_per_timestep) before node_emb if it's currently flattened.
        # However, if `x` comes in as (num_nodes * batch_size, in_channels_per_node_after_flattening_time),
        # then `in_channels` in `__init__` is correct as `in_timestep * (1+2)`.
        
        # The `x` input to the `forward` function of the model is already batched and flattened
        # by `DataLoader` and `Data` object if `num_graphs` > 1.
        # Its shape is (total_num_nodes_in_batch, in_channels)
        # Here, in_channels = in_timestep * (speed + time_of_day features) = 12 * (1 + 2) = 36
        
        x = self.node_emb(x) # (total_num_nodes_in_batch, hidden_channels - pe_dim or hidden_channels)
        
        if self.use_pe and pe is not None:
            pe = self.pe_lin(pe)
            x = torch.cat([x, pe], dim=-1) # (total_num_nodes_in_batch, hidden_channels)
            
        for layer in self.layers:
            x = layer(x, edge_index, edge_weight, batch)
            
        x = self.norm(x)
        x = self.mlp(x) # (total_num_nodes_in_batch, out_channels)
        
        batch_size = batch.max().item() + 1 if batch is not None else 1
        num_nodes = x.size(0) // batch_size # num_nodes for a single graph
        
        return x.view(batch_size, num_nodes, -1) # (batch_size, num_nodes, out_channels)

In [8]:
# Main workflow for single adjacency matrix
adj_matrices = [
    'adj_mx_knn_lane.pkl',
    'adj_mx_knn.pkl',
    'adj_mx_radius_lane.pkl',
    'adj_mx_radius.pkl'
]
results = {}

# Display available adjacency matrices
print("Available adjacency matrices:")
for i, adj_file in enumerate(adj_matrices, 1):
    print(f"{i}. {adj_file}")

# Prompt user for input
while True:
    try:
        choice = input("Enter the number (1-4) of the adjacency matrix to use: ")
        choice = int(choice)
        if 1 <= choice <= 4:
            adj_file = adj_matrices[choice - 1]
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 4.")
    except ValueError:
        print("Invalid input. Please enter a number between 1 and 4.")

print(f"\nProcessing with adjacency matrix: {adj_file}")

try:
    # Load data with the selected adjacency matrix
    node_values, edge_index, edge_weight, scaler, original_node_values_shape = load_metr_la_data(custom_adj_path=f'METR-LA/{adj_file}')
    
    # Define in_timestep and out_timestep
    in_timestep = 12
    out_timestep = 12

    # Create data samples with time features
    # features will have shape (num_samples, num_nodes, in_timestep * (speed_features + time_features))
    features, labels = create_data_samples(node_values, in_timestep=in_timestep, out_timestep=out_timestep)
    
    # Split data into train, validation, and test sets
    total_samples = len(features)
    train_end = int(0.6 * total_samples)
    val_end = train_end + int(0.2 * total_samples)
    
    train_features = features[:train_end]
    train_labels = labels[:train_end]
    val_features = features[train_end:val_end]
    val_labels = labels[train_end:val_end]
    test_features = features[val_end:]
    test_labels = labels[val_end:]
    
    print(f"Train samples: {len(train_features)}, Val samples: {len(val_features)}, Test samples: {len(test_features)}")
    
    # Apply random walk positional encoding
    transform = AddRandomWalkPE(walk_length=20, attr_name='pe')
    
    # Create training data
    train_data = []
    for i in range(len(train_features)):
        # x is (num_nodes, in_features_per_node)
        data = Data(x=train_features[i], y=train_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        train_data.append(data)
    
    # Create validation data
    val_data = []
    for i in range(len(val_features)):
        data = Data(x=val_features[i], y=val_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        val_data.append(data)
    
    # Create test data
    test_data = []
    for i in range(len(test_features)):
        data = Data(x=test_features[i], y=test_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        test_data.append(data)
    
    # Create 100th day data
    day_100_features, day_100_labels, day_100_scaler = create_100th_day_data(in_timestep=in_timestep, out_timestep=out_timestep)
    day_100_data = []
    for i in range(len(day_100_features)):
        data = Data(x=day_100_features[i], y=day_100_labels[i].T, edge_index=edge_index, edge_attr=edge_weight)
        data = transform(data)
        if 'pe' not in data:
            print(f"Warning: Positional encoding 'pe' not added to Day 100 Data {i}; proceeding without PE")
        day_100_data.append(data)
    
    # Create data loaders
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
    day_100_loader = DataLoader(day_100_data, batch_size=32, shuffle=False)
    
    # Determine the actual `in_channels` for the model after adding time features.
    # Each node at each input timestep now has 1 (speed) + 2 (time features) = 3 features.
    # Total `in_channels` for the node embedding layer is `in_timestep * 3`.
    model_in_channels = in_timestep * 3 

    # Initialize model
    model = GPSGraphTransformer(in_channels=model_in_channels, hidden_channels=64, out_channels=out_timestep, num_layers=4, pe_dim=20, heads=4, use_pe=True)
    model = model.to(device)
    
    # Optimizer and loss function
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    criterion = nn.MSELoss()
    
    # Training loop
    for epoch in range(20):
        model.train()
        total_train_loss = 0
        for data in train_loader:
            data = data.to(device)
            if data.edge_index.shape[0] != 2:
                raise ValueError(f"Invalid edge_index shape before model call: {data.edge_index.shape}")
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        train_loss = total_train_loss / len(train_loader)
        
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                if data.edge_index.shape[0] != 2:
                    raise ValueError(f"Invalid edge_index shape in val_loader: {data.edge_index.shape}")
                out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
                batch_size = data.batch.max().item() + 1
                num_nodes = out.shape[1]
                target = data.y.view(batch_size, num_nodes, -1)
                loss = criterion(out, target)
                total_val_loss += loss.item()
        val_loss = total_val_loss / len(val_loader)
        print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    # Evaluate on test set
    model.eval()
    total_mae, total_rmse, total_mape = 0, 0, 0
    num_batches = 0
    with torch.no_grad():
        for data in test_loader:
            data = data.to(device)
            if data.edge_index.shape[0] != 2:
                raise ValueError(f"Invalid edge_index shape in test_loader: {data.edge_index.shape}")
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            mae, rmse, mape = compute_metrics(out, target, scaler)
            total_mae += mae
            total_rmse += rmse
            total_mape += mape
            num_batches += 1
            if num_batches == 1:
                visualize_predictions(out, target, scaler, output_dir='plots_test', filename=f'prediction_test_{adj_file}_with_time.png')
        test_mae = total_mae / num_batches
        test_rmse = total_rmse / num_batches
        test_mape = total_mape / num_batches
        print(f"Test MAE: {test_mae:.4f} mph, RMSE: {test_rmse:.4f} mph, MAPE: {test_mape:.4f}%\n")
    
    # Evaluate on 100th day
    total_mae, total_rmse, total_mape = 0, 0, 0
    num_batches = 0
    with torch.no_grad():
        for data in day_100_loader:
            data = data.to(device)
            if data.edge_index.shape[0] != 2:
                raise ValueError(f"Invalid edge_index shape in day_100_loader: {data.edge_index.shape}")
            out = model(data.x, data.edge_index, data.edge_attr, data.pe if hasattr(data, 'pe') else None, data.batch)
            batch_size = data.batch.max().item() + 1
            num_nodes = out.shape[1]
            target = data.y.view(batch_size, num_nodes, -1)
            mae, rmse, mape = compute_metrics(out, target, day_100_scaler)
            total_mae += mae
            total_rmse += rmse
            total_mape += mape
            num_batches += 1
            if num_batches == 1:
                visualize_predictions(out, target, day_100_scaler, filename=f'prediction_100th_day_{adj_file}_with_time.png')
        day_100_mae = total_mae / num_batches
        day_100_rmse = total_rmse / num_batches
        day_100_mape = total_mape / num_batches
        print(f"100th Day MAE: {day_100_mae:.4f} mph, RMSE: {day_100_rmse:.4f} mph, MAPE: {day_100_mape:.4f}%\n")
    
    # Visualize full 100th day
    visualize_full_100th_day(day_100_loader, model, day_100_scaler, filename=f'full_100th_day_prediction_{adj_file}_with_time.png')
    
    # Store results
    results[adj_file] = {
        'test_mae': test_mae, 'test_rmse': test_rmse, 'test_mape': test_mape,
        'day_100_mae': day_100_mae, 'day_100_rmse': day_100_rmse, 'day_100_mape': day_100_mape
    }

except Exception as e:
    print(f"Error processing {adj_file}: {str(e)}")
    raise

# Print summary of results
print("\nSummary of Results:")
for adj_file, metrics in results.items():
    print(f"{adj_file}:")
    print(f"  Test - MAE: {metrics['test_mae']:.4f} mph, RMSE: {metrics['test_rmse']:.4f} mph, MAPE: {metrics['test_mape']:.4f}%")
    print(f"  100th Day - MAE: {metrics['day_100_mae']:.4f} mph, RMSE: {metrics['day_100_rmse']:.4f} mph, MAPE: {metrics['day_100_mape']:.4f}%")

Available adjacency matrices:
1. adj_mx_knn_lane.pkl
2. adj_mx_knn.pkl
3. adj_mx_radius_lane.pkl
4. adj_mx_radius.pkl

Processing with adjacency matrix: adj_mx_radius_lane.pkl
Loading custom adjacency matrix from: METR-LA/adj_mx_radius_lane.pkl
Number of non-zero entries in adjacency matrix: 282
Loaded adjacency matrix from METR-LA/adj_mx_radius_lane.pkl, shape: (207, 207), dtype: float32
Adjacency matrix shape: (207, 207), dtype: float32
Original node_values shape: (34272, 207, 2)
Processed node_values shape: (207, 28512)
Raw edge_index from np.where: 2 arrays, shapes: [(282,), (282,)]
Stacked edge_index shape: (2, 282)
edge_index device: cuda:0, shape: torch.Size([2, 282])
edge_weight device: cuda:0, shape: torch.Size([282])
Train samples: 17093, Val samples: 5697, Test samples: 5699
Epoch 1, Train Loss: 0.0538, Val Loss: 0.0243
Epoch 2, Train Loss: 0.0319, Val Loss: 0.0213
Epoch 3, Train Loss: 0.0291, Val Loss: 0.0208
Epoch 4, Train Loss: 0.0276, Val Loss: 0.0205
Epoch 5, Train Loss